In [5]:
import re
import json
from pypdf import PdfReader

def semantic_parse_factories_act(pdf_path):
    reader = PdfReader(pdf_path)
    full_text = ""
    
    # Extract raw text with page markers
    for page_num, page in enumerate(reader.pages):
        text = page.extract_text()
        if text:
            full_text += f"\n[PAGE_MARKER:{page_num + 1}]\n" + text

    # Pattern to catch section numbers like "21. Fencing of machinery.—" or "41A. Constitution of..."
    section_pattern = re.compile(r'(\n\d+[A-Z]?\.\s+[A-Z][a-z]+[^.\n—]+(?:.—|—|\.))')
    
    # Split text by section landmarks
    parts = section_pattern.split(full_text)
    
    chunks = []
    current_section = "Preamble / Preliminary Definitions"
    current_page = 1
    chunk_idx = 0
    
    # The first element before any split is the header/preamble
    first_part = parts[0]
    page_find = re.findall(r'\[PAGE_MARKER:(\d+)\]', first_part)
    if page_find: current_page = int(page_find[-1])
    clean_text = re.sub(r'\[PAGE_MARKER:\d+\]', '', first_part).strip()
    if clean_text:
        chunks.append({
            "text": clean_text,
            "metadata": {"source": "factories_act_1948.pdf", "page": current_page, "section_title": current_section, "chunk_id": f"chunk_{chunk_idx}"}
        })
        chunk_idx += 1

    # Loop through the matches and text blocks pairs
    for i in range(1, len(parts), 2):
        header = parts[i].replace('\n', ' ').strip()
        body = parts[i+1]
        
        # Determine the page number context
        page_find = re.findall(r'\[PAGE_MARKER:(\d+)\]', body)
        if page_find:
            current_page = int(page_find[0]) # Page where this block starts
            
        clean_body_text = re.sub(r'\[PAGE_MARKER:\d+\]', '', body).strip()
        combined_text = f"{header} {clean_body_text}"
        
        chunks.append({
            "text": combined_text,
            "metadata": {
                "source": "factories_act_1948.pdf",
                "page": current_page,
                "section_title": header.split('.—')[0].split('—')[0].strip(),
                "chunk_id": f"chunk_{chunk_idx}"
            }
        })
        chunk_idx += 1
        
    return chunks

# Run ingestion
pdf_path = "data/regulatory/factories_act_1948.pdf"
docs = semantic_parse_factories_act(pdf_path)

# Print exactly 5 sample chunks to eyeball text integrity
print(f"🔥 Total Semantic Chunks Generated: {len(docs)}\n")
for d in docs[15:20]: # Viewing a slice containing core safety protocols
    print(f"🔹 [{d['metadata']['chunk_id']}] | Section: {d['metadata']['section_title']} | Page: {d['metadata']['page']}")
    print(f"Text snippet: {d['text'][:280]}...\n" + "-"*50)

🔥 Total Semantic Chunks Generated: 451

🔹 [chunk_15] | Section: 14. Dust and fume. | Page: 1
Text snippet: 14. Dust and fume. ...
--------------------------------------------------
🔹 [chunk_16] | Section: 15. Artificial humidification. | Page: 1
Text snippet: 15. Artificial humidification. ...
--------------------------------------------------
🔹 [chunk_17] | Section: 16. Overcrowding. | Page: 1
Text snippet: 16. Overcrowding. 17. L ighting....
--------------------------------------------------
🔹 [chunk_18] | Section: 18. Drinking water. | Page: 1
Text snippet: 18. Drinking water. ...
--------------------------------------------------
🔹 [chunk_19] | Section: 19. Latrines and urinals. | Page: 1
Text snippet: 19. Latrines and urinals. ...
--------------------------------------------------


In [1]:
from sentence_transformers import SentenceTransformer

print("⏳ Downloading all-MiniLM-L6-v2 directly to your local cache...")
# This will pull down the ~90MB PyTorch-backed weights explicitly
model = SentenceTransformer("all-MiniLM-L6-v2")
print("✅ Model successfully downloaded and cached locally!")

e:\AI-for-industrial-intelligence\.venv\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


⏳ Downloading all-MiniLM-L6-v2 directly to your local cache...
✅ Model successfully downloaded and cached locally!


In [2]:
import chromadb
from chromadb.utils import embedding_functions

# Initialize local PyTorch-backed HuggingFace embedding engine
local_emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

# Create persistent local storage client 
chroma_client = chromadb.PersistentClient(path="./data/vectorstore/")
collection = chroma_client.get_or_create_collection(name="industrial_brain_v1", embedding_function=local_emb_fn)

# Clear old vector runs if re-running the cell
if collection.count() > 0:
    all_ids = collection.get()["ids"]
    collection.delete(ids=all_ids)

# Load chunks into Chroma
collection.add(
    documents=[d["text"] for d in docs],
    metadatas=[d["metadata"] for d in docs],
    ids=[d["metadata"]["chunk_id"] for d in docs]
)

print(f"💾 Local ChromaDB successfully loaded with {collection.count()} persistent records.")

e:\AI-for-industrial-intelligence\.venv\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionDeleteEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


💾 Local ChromaDB successfully loaded with 451 persistent records.


In [3]:
def run_manual_retrieval(query_str):
    results = collection.query(query_texts=[query_str], n_results=5)
    print(f"\n==================================================")
    print(f"🔍 SEARCH QUERY: '{query_str}'")
    print(f"==================================================")
    
    for idx in range(len(results['documents'][0])):
        chunk_id = results['ids'][0][idx]
        score = results['distances'][0][idx]
        meta = results['metadatas'][0][idx]
        text_content = results['documents'][0][idx]
        
        print(f"📍 Match {idx+1} | ID: {chunk_id} | Distance Score: {score:.4f}")
        print(f"   Source: {meta['source']} | Section: {meta['section_title']} | Page: {meta['page']}")
        print(f"   Text: {text_content[:200]}...\n")

# Run tests
run_manual_retrieval("What safety measures apply to fencing of machinery?")
run_manual_retrieval("What are the provisions for hazardous processes?")
run_manual_retrieval("confined space entry requirements")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



🔍 SEARCH QUERY: 'What safety measures apply to fencing of machinery?'
📍 Match 1 | ID: chunk_21 | Distance Score: 0.5580
   Source: factories_act_1948.pdf | Section: 21. Fencing of machinery. | Page: 1
   Text: 21. Fencing of machinery. ...

📍 Match 2 | ID: chunk_206 | Distance Score: 0.6723
   Source: factories_act_1948.pdf | Section: 21. Fencing of machinery | Page: 18
   Text: 21. Fencing of machinery .— (1) In every factory the following, namely: — 
(i) every moving part of a prime mover and every flywheel connected to a prime mover, whether 
the prime m over or flywheel i...

📍 Match 3 | ID: chunk_40 | Distance Score: 0.8264
   Source: factories_act_1948.pdf | Section: 40. Safety of buildings and machinery. | Page: 2
   Text: 40. Safety of buildings and machinery. ...

📍 Match 4 | ID: chunk_234 | Distance Score: 0.9806
   Source: factories_act_1948.pdf | Section: 33. Pits, sumps openings in floors, etc | Page: 23
   Text: 33. Pits, sumps openings in floors, etc .— (1) In every fac

In [9]:
import os
from litellm import completion

# Set your Gemini API key in the environment so LiteLLM can find it
# os.environ["GEMINI_API_KEY"] = "your-actual-api-key-here"

def test_grounded_generation(query_str):
    # 1. Fetch Top 3 chunks for strict contextual limits
    retrieved = collection.query(query_texts=[query_str], n_results=3)
    
    context_blocks = []
    for i in range(len(retrieved['documents'][0])):
        c_id = retrieved['ids'][0][i]
        sec = retrieved['metadatas'][0][i]['section_title']
        p_num = retrieved['metadatas'][0][i]['page']
        txt = retrieved['documents'][0][i]
        context_blocks.append(f"--- ID: {c_id} (Section: {sec}, Page: {p_num}) ---\n{txt}")
    
    full_context_str = "\n\n".join(context_blocks)
    
    system_prompt = (
        "You are an industrial compliance inspector. Synthesize an answer for the query using ONLY the provided text blocks. "
        "Every single factual claim you make must end with a precise inline bracket citation matching the source ID exactly (e.g. [chunk_12]). "
        "If the provided context chunks do not contain enough specific facts to fully answer a question, explicitly state: "
        "'INFO NOT IN CONTEXT'. Do not assume, complete, or pull external engineering rules."
    )
    
    user_prompt = f"CONTEXT CHUNKS:\n{full_context_str}\n\nQUERY: {query_str}"
    
    # 2. Execute single LLM call routed through LiteLLM to Gemini
    response = completion(
        model="gemini/gemini-3.5-flash", # You can also use "gemini/gemini-1.5-flash" for faster/cheaper responses
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.0
    )
    
    answer = response.choices[0].message.content
    
    print(f"\n==================================================")
    print(f"🤖 LLM RESPONSE GENERATION FOR: '{query_str}'")
    print(f"==================================================")
    print(f"📋 Prompt Context Shared:\n{full_context_str[:400]}...\n[Truncated for Output]\n")
    print(f"✍️ Generated Output:\n{answer}\n")
    print(f"⚠️ MANUAL AUDIT NOTE: Cross-reference the inline brackets to confirm no ungrounded facts leaked through.")

# Run generation checks
test_grounded_generation("What safety measures apply to fencing of machinery?")
test_grounded_generation("What are the provisions for hazardous processes?")
test_grounded_generation("confined space entry requirements")


🤖 LLM RESPONSE GENERATION FOR: 'What safety measures apply to fencing of machinery?'
📋 Prompt Context Shared:
--- ID: chunk_21 (Section: 21. Fencing of machinery., Page: 1) ---
21. Fencing of machinery. 

--- ID: chunk_206 (Section: 21. Fencing of machinery, Page: 18) ---
21. Fencing of machinery .— (1) In every factory the following, namely: — 
(i) every moving part of a prime mover and every flywheel connected to a prime mover, whether 
the prime m over or flywheel is in the engine house or not;  
(ii) ...
[Truncated for Output]

✍️ Generated Output:
Based on the provided compliance text, the safety measures that apply to the fencing of machinery in a factory include the following:

*   **Mandatory Fencing of Specific Machinery**: The following parts must be securely fenced by safeguards of substantial construction:
    *   Every moving part of a prime mover and every flywheel connected to a prime mover, regardless of whether it is located in the engine house [chunk_206].
    *   Th